# Mesh 4 · Changing a mesh

**Fundamentals, part 4 of 4.** Companion to [Week 05 §9](../README.md#9-changing-a-mesh).
Do [`03_data_on_a_mesh`](03_data_on_a_mesh.ipynb) first.

Notebook 03 moved vertices: geometry changed, connectivity did not. This
one does the opposite kind of edit, the kind that **adds and removes
elements**. Each operation is followed by the same three questions:
how many vertices, edges and faces now, and is `V − E + F` still what it
should be?

| You will | Section |
| --- | --- |
| learn which edits change the mesh in place | 1 |
| triangulate | 2 |
| open a hole, and close one | 3 |
| refine locally: insert, split, collapse | 4 |
| refine everywhere: subdivision | 5 |
| swap faces and vertices: the dual | 6 |
| join two meshes properly | 7 |

In [ ]:
# Setup: run this cell first. It finds the shared drawing helper (viz.py),
# whether this notebook sits in its lecture folder or is your copy in MyWork/.
import sys
import pathlib

HERE = pathlib.Path.cwd()
NB = next(p / "Lecture" / "notebooks" for p in [HERE, *HERE.parents]
          if (p / "Lecture" / "notebooks" / "viz.py").exists())
sys.path.insert(0, str(NB))
from viz import chart, draw_graph, draw_mesh, draw_plan, draw_points, ramp, row, view3d

from compas.datastructures import Mesh


def grid(nx, ny, spacing=1.0):
    """The grid from notebook 01: keys along x first, every face counter-clockwise."""
    mesh = Mesh()
    key = {}
    for j in range(ny + 1):
        for i in range(nx + 1):
            key[i, j] = mesh.add_vertex(x=i * spacing, y=j * spacing, z=0)
    for j in range(ny):
        for i in range(nx):
            mesh.add_face([key[i, j], key[i + 1, j], key[i + 1, j + 1], key[i, j + 1]])
    return mesh


def counts(mesh):
    return (f"V {mesh.number_of_vertices():3d}  E {mesh.number_of_edges():3d}  "
            f"F {mesh.number_of_faces():3d}  euler {mesh.euler()}")

## 1 · In place, or a new mesh? Know before you call

![The grid; triangulated; with a face deleted; with a vertex inserted in a face. Counts and Euler under each](../figures/mesh-operations.svg)

Some methods **change the mesh you call them on** and return nothing
useful. Others leave it alone and **return a new one**. Mixing them up is
the most common way to lose a mesh you still needed.

| Changes the mesh in place | Returns a new mesh |
| --- | --- |
| `quads_to_triangles()` · `delete_face(f)` · `insert_vertex(f)` · `split_edge(e)` · `collapse_edge(e)` · `weld()` · `join(other)` | `copy()` · `subdivided(...)` · `dual()` · `thickened(...)` |

The habit: **copy first**, edit the copy.

In [ ]:
base = grid(3, 3)
print("the grid:           ", counts(base))

## 2 · Triangulate

Every quad gets a diagonal. Vertices stay; edges and faces grow.

In [ ]:
tri = base.copy()
tri.quads_to_triangles()
print("quads_to_triangles: ", counts(tri))
print("the original is untouched:", counts(base))
row(draw_mesh(base, "quads", vertex_labels=False, size=(300, 300)),
    draw_mesh(tri, "triangles", vertex_labels=False, size=(300, 300)))

Nine faces became eighteen, nine edges were added, and Euler did not move:
`16 − 33 + 18 = 1`. A diagonal adds one edge and one face, and
`− 1 + 1 = 0`. **Any edit that keeps the surface in one piece without
holes keeps Euler.** That is the check for everything below.

Why triangulate? Three points are always flat (panels, notebook 05), and
a triangle cannot shear (trusses, notebook 08).

## 3 · A hole

Delete a face. This is the one edit here that *should* change Euler.

In [ ]:
holed = base.copy()
holed.delete_face(4)
print("delete_face(4):     ", counts(holed))
print("vertex 5 on the boundary before:", base.is_vertex_on_boundary(5), "| after:", holed.is_vertex_on_boundary(5))
inner = [v for v in holed.vertices_on_boundary() if not base.is_vertex_on_boundary(v)]
draw_mesh(holed, "one face fewer: euler 0", highlight_vertices=inner, size=(380, 340))

The count went 9 → 8 faces and nothing else, so Euler dropped from 1 to 0:
**a sheet with one hole**. The four vertices round it are now boundary
vertices, though nobody moved them. A skylight, a door, a missing scan
patch: all the same edit, and all visible in one integer. Notebook 09
uses exactly this to decide whether a scanned roof was trimmed correctly.

Closing it again is adding the face back:

In [ ]:
holed.add_face([5, 6, 10, 9])
print("face added back:    ", counts(holed))

## 4 · Local refinement: more detail where it is needed

Three small edits. Each keeps Euler at 1.

In [ ]:
fan = base.copy()
new = fan.insert_vertex(4)
print(f"insert_vertex(4):    {counts(fan)}   new vertex {new}")

cut = base.copy()
a, b = cut.split_face(4, 5, 10)
print(f"split_face(4, 5, 10): {counts(cut)}   face 4 became faces {a} and {b}")

longer = base.copy()
mid = longer.split_edge((5, 6), t=0.5, allow_boundary=True)
print(f"split_edge((5, 6)):  {counts(longer)}   new vertex {mid}")

shorter = base.copy()
shorter.collapse_edge((5, 6), t=0.5, allow_boundary=True)
print(f"collapse_edge((5, 6)): {counts(shorter)}")

row(draw_mesh(fan, "insert_vertex(4)", size=(290, 290), highlight_vertices=[new]),
    draw_mesh(cut, "split_face(4, 5, 10)", size=(290, 290), highlight_edges=[(5, 10)]),
    draw_mesh(longer, "split_edge((5, 6))", size=(290, 290), highlight_vertices=[mid]),
    draw_mesh(shorter, "collapse_edge((5, 6))", size=(290, 290)))

Look at the third and fourth pictures: after `split_edge` the two faces
beside that edge have **five** corners, and after `collapse_edge` they
have **three**. A mesh does not require every face to have the same number
of sides. Whether your *fabrication* does is a different question, and
`face_vertices` answers it:

In [ ]:
for name, m in (("split_edge", longer), ("collapse_edge", shorter)):
    sides = sorted(len(m.face_vertices(face)) for face in m.faces())
    print(f"{name:14s} sides per face: {sides}")

## 5 · Refinement everywhere: subdivision

![A cube subdivided once and twice with Catmull-Clark; Euler stays 2](../figures/mesh-subdivide.svg)

Subdivision splits **every** face and, depending on the scheme, moves the
vertices to smooth the result. It returns a new mesh.

In [ ]:
import math

cube = Mesh.from_polyhedron(6)


def reach(mesh):
    """How far the farthest vertex is from the centre."""
    return max(math.dist(mesh.vertex_coordinates(v), [0, 0, 0]) for v in mesh.vertices())


print(f"{'the cube':32s} {counts(cube)}   reach {reach(cube):.2f}")
for scheme in ("quad", "catmullclark", "tri", "corner"):
    for k in (1, 2):
        m = cube.subdivided(scheme=scheme, k=k)
        print(f'subdivided("{scheme}", k={k})'.ljust(32), counts(m), f"  reach {reach(m):.2f}")

Read the last column: how far the farthest vertex is from the centre.
`quad`, `tri` and `corner` leave it at 2.00: the corners have not moved,
and the cube is still a cube with more faces. `catmullclark` pulls them in,
further each round: the cube is on its way to a sphere. Euler is 2 in
every row, because a closed box stays a closed box.

Two uses, then. **Refine without changing the shape** when a calculation
needs more points. **Smooth a coarse cage** when you want to design with
eight points instead of four hundred (notebook 06 does this to a shell).

In [ ]:
view3d(cube.subdivided(scheme="catmullclark", k=3))

## 6 · The dual: faces become vertices

![The dual of a 4 by 3 grid: a vertex at each face centre](../figures/mesh-dual.svg)

Put a point in every face; connect the points of neighbouring faces. A
vertex where *n* faces met becomes a face with *n* sides.

In [ ]:
tri_grid = grid(4, 4)
tri_grid.quads_to_triangles()
dual = tri_grid.dual()
print("triangles:", counts(tri_grid))
print("its dual: ", counts(dual))
print("sides per dual face:", sorted({len(dual.face_vertices(face)) for face in dual.faces()}))
row(draw_mesh(tri_grid, "triangles", vertex_labels=False, size=(360, 360)),
    draw_mesh(dual, "their dual", vertex_labels=False, size=(360, 360)))

Inner vertices of this triangle grid have six neighbours, so the dual is
**hexagons**: a honeycomb, from a grid of triangles, in one call. The
pairing runs through architecture: a triangulated truss and the cells it
encloses, a Delaunay triangulation and its Voronoi diagram (Week 10), a
force diagram and its form diagram.

(By default `dual()` leaves the boundary out, which is why it is smaller
than the mesh it came from. `dual(include_boundary=True)` keeps it.)

## 7 · Joining two meshes

`join` puts two meshes in one object. It does **not** connect them.

In [ ]:
left, right = grid(2, 2), grid(2, 2)
for v in right.vertices():
    right.vertex_attribute(v, "x", right.vertex_attribute(v, "x") + 2)

left.join(right)
print("joined:  ", counts(left), "  <- euler 2: still two sheets")
left.weld()
print("welded:  ", counts(left), "  <- euler 1: one sheet")
draw_mesh(left, "two grids, joined and welded", face_labels=True, size=(520, 300))

Notebook 01's duplicated-vertex bug, met again from the other side. After
`join`, the seam has two vertices at every point, and no face on the left
is a neighbour of any face on the right. `weld` merges them. **Euler told
you which state you were in, both times.**

## What to keep

- Copy first. Know which methods edit in place.
- After every edit: counts, and Euler.
- Euler **stays** through triangulating, splitting, collapsing,
  subdividing. It **changes** when you open a hole or fail to weld.
- Faces need not all have the same number of sides. Check, if it matters.

That is the end of the fundamentals. The notebooks from 05 on use them on
buildings: [`05_canopy_panels`](05_canopy_panels.ipynb) is the gentlest.

## Your turn
1. Delete **two** faces of `grid(4, 4)` that do not touch. Predict Euler.
   Now delete two that share an edge. Predict again.
2. `insert_vertex` in every face of `grid(3, 3)`. What are the sides per
   face, and the degrees of the old and new vertices?
3. Subdivide a tetrahedron (`Mesh.from_polyhedron(4)`) with
   `catmullclark`, `k=3`. What shape is it approaching? Is it all quads?
4. Take the dual of the dual of `grid(4, 4)`. Do you get the grid back?